<a href="https://colab.research.google.com/github/joelmanrique91-lgtm/geostats-colab-lab/blob/main/KNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de Componentes Principales (PCA) Geoquímico / Mineralógico Composiciónal Optimizado

Este notebook interactivo ha sido optimizado para Google Colab, mejorando la estructura del código, la claridad y añadiendo interactividad para algunos parámetros clave.

## 1. Instalación de Librerías

In [ ]:
# Instalar librerías necesarias. 'kneed' no se utiliza en el código original, por lo que se omite.

In [ ]:
!pip install -q pandas numpy scipy scikit-learn plotly seaborn openpyxl hdbscan reportlab

## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os # Added for file system operations
import tempfile # Added for creating temporary files

from scipy.stats import gmean

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.covariance import EmpiricalCovariance

import hdbscan

import plotly.express as px
import plotly.graph_objects as go

import seaborn as sns
import matplotlib.pyplot as plt

from google.colab import files

import ipywidgets as widgets
from IPython.display import display, clear_output

# PDF generation libraries
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image # Added for images
)
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter, A4 # Added A4 for more options
from reportlab.lib.units import inch # Added for unit conversion
from reportlab.lib.utils import ImageReader # Added for image reading


## 3. Carga de Archivo CSV

In [ ]:
print("\nSUBIR ARCHIVO CSV\n")

try:
    uploaded = files.upload()
    if not uploaded:
        print("No se ha subido ningún archivo.")
        df = pd.DataFrame() # Initialize empty DataFrame
    else:
        filename = list(uploaded.keys())[0]
        df = pd.read_csv(filename)

        print("\nDATASET CARGADO\n")
        display(df.head())

        print("\nSHAPE:")
        print(df.shape)

        print("\nDTYPES:")
        display(df.dtypes)

        print("\nNULLS (%):")
        display(
            (df.isnull().mean() * 100)
            .sort_values(ascending=False)
        )
except Exception as e:
    print(f"Error al cargar el archivo: {e}")
    df = pd.DataFrame() # Initialize empty DataFrame in case of error

## 4. Detección Automática de Columnas Numéricas

In [ ]:
if not df.empty:
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    print("\nCOLUMNAS NUMÉRICAS DETECTADAS:")
    print(numeric_cols)
else:
    numeric_cols = []
    print("DataFrame vacío. No se pudieron detectar columnas numéricas.")

## 5. Selectores Interactivos para PCA y Visualización

In [ ]:
print("\nSELECCIONAR VARIABLES PARA PCA\n")

multi_select = widgets.SelectMultiple(
    options=numeric_cols if numeric_cols else ['No numeric columns found'],
    value=numeric_cols if numeric_cols else [],
    rows=min(len(numeric_cols), 25) if numeric_cols else 1,
    description='Variables para PCA',
    layout=widgets.Layout(width='80%', height='350px'),
    disabled=df.empty
)

all_cols_for_dropdown = ["NONE"] + list(df.columns) if not df.empty else ["NONE"]

lith_widget = widgets.Dropdown(
    options=all_cols_for_dropdown,
    value="NONE",
    description="Litología",
    disabled=df.empty
)

alter_widget = widgets.Dropdown(
    options=all_cols_for_dropdown,
    value="NONE",
    description="Alteración",
    disabled=df.empty
)

depth_widget = widgets.Dropdown(
    options=all_cols_for_dropdown,
    value="NONE",
    description="Profundidad",
    disabled=df.empty
)

domain_widget = widgets.Dropdown(
    options=all_cols_for_dropdown,
    value="NONE",
    description="Dominio",
    disabled=df.empty
)

# Grouping widgets for better display
display(multi_select)

print("\nVARIABLES OPCIONALES PARA VISUALIZACIÓN\n")
display(lith_widget)
display(alter_widget)
display(depth_widget)
display(domain_widget)

## 6. Configuración de PCA y Clustering

In [ ]:
epsilon_widget = widgets.FloatText(
    value=1e-6,
    description='Epsilon (para ceros)',
    disabled=df.empty
)

ncomp_widget = widgets.IntSlider(
    value=5,
    min=2,
    max=min(15, len(numeric_cols) if numeric_cols else 2),
    step=1,
    description='N° Componentes Principales',
    disabled=df.empty
)

kmeans_clusters_widget = widgets.IntSlider(
    value=4,
    min=2,
    max=10, # Arbitrary max, could be dynamic based on data size
    step=1,
    description='N° Clusters K-Means',
    disabled=df.empty
)

hdbscan_min_cluster_size_widget = widgets.IntSlider(
    value=50,
    min=2,
    max=200, # Arbitrary max
    step=1,
    description='HDBSCAN Min Cluster Size',
    disabled=df.empty
)

display(epsilon_widget)
display(ncomp_widget)
display(kmeans_clusters_widget)
display(hdbscan_min_cluster_size_widget)

## 7. Generador de Reporte PDF

In [ ]:
def export_pdf_report(
    explained_df,
    loadings_df,
    score_df,
    image_paths=[], # New parameter to accept image paths
    output_name="PCA_Geological_Report.pdf"
):

    print("\nGENERANDO PDF...\n")

    # Use A4 page size for better international compatibility
    doc = SimpleDocTemplate(
        output_name,
        pagesize=A4,
        rightMargin=0.5*inch,
        leftMargin=0.5*inch,
        topMargin=0.5*inch,
        bottomMargin=0.5*inch
    )

    styles = getSampleStyleSheet()
    # Define a custom style for smaller tables to fit better
    styles.add(
        ParagraphStyle(
            name='SmallerBodyText',
            parent=styles['BodyText'],
            fontSize=9,
            leading=11
        )
    )
    styles.add(
        ParagraphStyle(
            name='SmallerHeading3',
            parent=styles['Heading3'],
            fontSize=10,
            leading=12
        )
    )

    elements = []

    # ========================================================
    # TÍTULO
    # ========================================================

    elements.append(
        Paragraph(
            "<b>Análisis de Componentes Principales (PCA) Mineralógico / Geoquímico</b>",
            styles['Title']
        )
    )

    elements.append(Spacer(1, 20))

    # ========================================================
    # RESUMEN
    # ========================================================

    summary = f"""
    <b>Resumen del Análisis</b><br/><br/>
    Muestras Analizadas: {score_df.shape[0]}<br/>
    Variables Geoquímicas/Mineralógicas: {loadings_df.shape[0]}<br/>
    Componentes Principales (PCs) Calculadas: {loadings_df.shape[1]}<br/>
    """

    elements.append(
        Paragraph(summary, styles['BodyText'])
    )

    elements.append(Spacer(1, 20))

    # ========================================================
    # VARIANZA EXPLICADA
    # ========================================================

    elements.append(
        Paragraph(
            "<b>Varianza Explicada por Componente Principal</b>",
            styles['Heading2']
        )
    )

    table_data = [explained_df.columns.tolist()]

    for _, row in explained_df.iterrows():
        table_data.append(
            [f"{x:.2f}" if isinstance(x, (int, float)) else str(x) for x in row.values] # Format numbers for display
        )

    # Adjust font size for table content
    table_style = TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.grey),
        ('TEXTCOLOR', (0,0), (-1,0), colors.white),
        ('GRID', (0,0), (-1,-1), 1, colors.black),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 9) # Smaller font for table content
    ])
    table = Table(table_data)
    table.setStyle(table_style)
    elements.append(table)

    elements.append(Spacer(1, 25))

    # ========================================================
    # PLOTS DE ANÁLISIS
    # ========================================================

    elements.append(
        Paragraph(
            "<b>Visualizaciones Clave del PCA</b>",
            styles['Heading2']
        )
    )
    elements.append(Spacer(1, 10))

    for img_path in image_paths:
        try:
            img = Image(img_path)
            # Scale image to fit within page width, maintaining aspect ratio
            img_width, img_height = img.drawWidth, img.drawHeight
            max_width = A4[0] - 2 * 0.5*inch # Page width minus margins
            if img_width > max_width:
                scale_factor = max_width / img_width
                img.drawWidth = img_width * scale_factor
                img.drawHeight = img_height * scale_factor

            elements.append(img)
            elements.append(Spacer(1, 10))
        except Exception as e:
            elements.append(
                Paragraph(
                    f"<i>No se pudo cargar la imagen: {os.path.basename(img_path)} ({e})</i>",
                    styles['Error']
                )
            )
            elements.append(Spacer(1, 5))

    elements.append(Spacer(1, 25))

    # ========================================================
    # PRINCIPALES LOADINGS
    # ========================================================

    elements.append(
        Paragraph(
            "<b>Variables con Mayores Loadings por Componente Principal</b>",
            styles['Heading2']
        )
    )

    for pc in loadings_df.columns:

        elements.append(
            Paragraph(
                f"<b>{pc}</b>",
                styles['SmallerHeading3']
            )
        )

        top_loadings = (
            loadings_df[pc]
            .abs()
            .sort_values(ascending=False)
            .head(10)
        )

        pc_data = [["Variable", "Loading"]]

        for var in top_loadings.index:

            val = loadings_df.loc[var, pc]

            pc_data.append([
                var,
                f"{val:.4f}"
            ])

        pc_table = Table(pc_data)

        pc_table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), colors.darkblue),
            ('TEXTCOLOR', (0,0), (-1,0), colors.white),
            ('GRID', (0,0), (-1,-1), 1, colors.black),
            ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
            ('FONTSIZE', (0,0), (-1,-1), 8) # Smaller font for loading tables
        ]))

        elements.append(pc_table)

        elements.append(Spacer(1, 15))

    # ========================================================
    # CONCLUSIONES
    # ========================================================

    elements.append(
        Paragraph(
            "<b>Consideraciones y Conclusiones</b>",
            styles['Heading2']
        )
    )

    conclusions = """
    El Análisis de Componentes Principales (PCA) es una técnica robusta para la reducción de dimensionalidad y la identificación de patrones. En este estudio, permitió identificar asociaciones geoquímicas/mineralógicas y gradientes estadísticos latentes dentro del conjunto de datos proporcionado.<br/><br/>

    Las componentes principales, aunque matemáticamente derivadas, ofrecen una base sólida para la interpretación geológica cuando se validan con información contextual como la litología, el tipo de alteración y la distribución espacial de las muestras. Es crucial recordar que la validez de estas interpretaciones depende en gran medida de la calidad y representatividad de los datos de entrada.<br/><br/>

    Se recomienda encarecidamente realizar una revisión crítica de los resultados, prestando especial atención a:<br/>
    <ul>
        <li>La <b>dominancia litológica</b> y su influencia en la composición geoquímica/mineralógica.</li>
        <li>La identificación de <b>minerales o elementos dominantes</b> que definen cada componente principal.</li>
        <li>La <b>estabilidad del modelo PCA</b> ante variaciones en el conjunto de datos o la presencia de outliers.</li>
        <li>La <b>sensibilidad del análisis a valores atípicos</b> y cómo estos podrían estar afectando la estructura de las componentes.</li>
    </ul>
    Esta validación cruzada con el conocimiento geológico es fundamental para transformar los patrones estadísticos en inferencias significativas para la exploración o modelamiento geológico.
    """

    elements.append(
        Paragraph(conclusions, styles['BodyText'])
    )

    doc.build(elements)

    print(f"\nPDF EXPORTADO: {output_name}")

    files.download(output_name)

    # Clean up temporary image files
    for img_path in image_paths:
        try:
            os.remove(img_path)
        except OSError as e:
            print(f"Error al eliminar archivo temporal {img_path}: {e}")


## 8. Funciones Auxiliares para PCA

In [ ]:
def _perform_qaqc(data_frame, epsilon):
    """Realiza control de calidad y preprocesamiento de datos."""
    print("===================================")
    print("QA/QC y Preprocesamiento")
    print("===================================")

    print("\nVALORES NEGATIVOS:")
    neg_count = (data_frame < 0).sum().sum()
    print(f"{neg_count} valores negativos encontrados.")

    print("\nNULLS:")
    null_count_before_drop = data_frame.isnull().sum().sum()
    print(f"{null_count_before_drop} valores nulos encontrados antes de la eliminación.")

    # Handle NaNs: Drop rows with any NaN values in the selected columns
    initial_rows = data_frame.shape[0]
    data_frame = data_frame.dropna()
    rows_after_drop = data_frame.shape[0]
    if initial_rows > rows_after_drop:
        print(f"{initial_rows - rows_after_drop} filas eliminadas debido a valores nulos.")
    else:
        print("No se eliminaron filas por valores nulos.")

    print("\nCOLUMNAS CONSTANTES (eliminadas si existen):")
    constant_cols = data_frame.columns[data_frame.nunique() <= 1].tolist()
    if constant_cols:
        print(f"Columnas eliminadas: {constant_cols}")
        data_frame = data_frame.drop(columns=constant_cols)
    else:
        print("No se encontraron columnas constantes.")

    # Manejo de ceros
    zero_count = (data_frame == 0).sum().sum()
    if zero_count > 0:
        print(f"\n{zero_count} ceros reemplazados por epsilon ({epsilon}).")
        data_frame = data_frame.replace(0, epsilon)
    else:
        print("\nNo se encontraron ceros para reemplazar.")

    return data_frame

def _apply_clr_scaling(data_frame):
    """Aplica cierre composicional (CLR) y escalado estándar."""
    print("\nAplicando Cierre Composicional (CLR) y Escalado Estándar...")
    row_sums = data_frame.sum(axis=1)
    X_closed = data_frame.div(row_sums, axis=0)

    gm = gmean(X_closed, axis=1)
    X_clr = np.log(X_closed.div(gm, axis=0))

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clr)

    return X_scaled, scaler # Return scaler for potential inverse transform

def _execute_pca(X_scaled, n_components, mineral_cols):
    """Ejecuta el análisis de componentes principales (PCA)."""
    print("\nEjecutando PCA...")
    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(X_scaled)

    loadings = pd.DataFrame(
        pca.components_.T,
        columns=[f"PC{i+1}" for i in range(n_components)],
        index=mineral_cols
    )

    explained = pd.DataFrame({
        "PC": [f"PC{i+1}" for i in range(n_components)],
        "Eigenvalue": pca.explained_variance_,
        "Variance_%": pca.explained_variance_ratio_ * 100,
        "Cumulative_%": np.cumsum(
            pca.explained_variance_ratio_
        ) * 100
    })
    print("\n===================================")
    print("VARIANZA EXPLICADA")
    print("===================================\n")
    display(explained)

    return scores, loadings, explained, pca

def _prepare_score_df(scores, n_components, original_df, optional_widgets):
    """Prepara el DataFrame de scores con columnas opcionales."""
    score_df = pd.DataFrame(
        scores,
        columns=[f"PC{i+1}" for i in range(n_components)]
    )

    # Add optional columns if selected and exist in original_df
    # Ensure original_df has enough rows corresponding to score_df after QA/QC
    # This assumes the indices align or that original_df was filtered identically
    # For now, let's assume original_df is the full 'df' from global scope.
    # A more robust solution would pass the 'processed' original_df.
    original_df_filtered = original_df.iloc[original_df.index.isin(score_df.index if hasattr(score_df, 'index') else original_df.index)].reset_index(drop=True)

    if optional_widgets['lithology'] != "NONE" and optional_widgets['lithology'] in original_df_filtered.columns:
        score_df["LITHOLOGY"] = original_df_filtered[optional_widgets['lithology']]
    if optional_widgets['alteration'] != "NONE" and optional_widgets['alteration'] in original_df_filtered.columns:
        score_df["ALTERATION"] = original_df_filtered[optional_widgets['alteration']]
    if optional_widgets['depth'] != "NONE" and optional_widgets['depth'] in original_df_filtered.columns:
        score_df["DEPTH"] = original_df_filtered[optional_widgets['depth']]
    if optional_widgets['domain'] != "NONE" and optional_widgets['domain'] in original_df_filtered.columns:
        score_df["DOMAIN"] = original_df_filtered[optional_widgets['domain']]

    return score_df

def _generate_plots(score_df, explained_df, loadings_df, mineral_cols):
    """Genera y muestra las visualizaciones del PCA, guardándolas en archivos temporales."""
    print("\nGenerando Plots de PCA...")
    image_paths = [] # List to store paths of saved images

    # Scree Plot
    fig_scree = go.Figure()
    fig_scree.add_trace(go.Scatter(
        x=explained_df["PC"],
        y=explained_df["Eigenvalue"],
        mode='lines+markers'
    ))
    fig_scree.add_hline(
        y=1,
        line_dash="dash",
        line_color="red"
    )
    fig_scree.update_layout(
        title="Scree Plot",
        xaxis_title="Componentes",
        yaxis_title="Eigenvalue"
    )
    fig_scree.show()
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_file:
        fig_scree.write_image(tmp_file.name)
        image_paths.append(tmp_file.name)

    # Score Plot PC1 vs PC2
    color_col_score = None
    for c in ["ALTERATION", "LITHOLOGY", "DOMAIN"]:
        if c in score_df.columns:
            color_col_score = c
            break

    fig_score = px.scatter(
        score_df,
        x="PC1",
        y="PC2",
        color=color_col_score,
        hover_data=score_df.columns,
        title="Score Plot PC1 vs PC2"
    )
    fig_score.show()
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_file:
        fig_score.write_image(tmp_file.name)
        image_paths.append(tmp_file.name)

    # Biplot (PC1 vs PC2)
    fig_biplot = go.Figure()
    fig_biplot.add_trace(go.Scatter(
        x=score_df["PC1"],
        y=score_df["PC2"],
        mode='markers',
        marker=dict(size=4),
        name='Samples'
    ))
    scale = 4 # Scaling factor for loadings visualization
    if loadings_df.shape[1] >= 2: # Ensure there are at least two PCs for biplot
        for i, mineral in enumerate(mineral_cols):
            fig_biplot.add_trace(go.Scatter(
                x=[0, loadings_df.iloc[i,0] * scale],
                y=[0, loadings_df.iloc[i,1] * scale],
                mode='lines+text',
                text=["", mineral],
                textposition="top center",
                name=mineral
            ))
    fig_biplot.update_layout(
        title="Biplot (PC1 vs PC2)",
        xaxis_title="PC1",
        yaxis_title="PC2"
    )
    fig_biplot.show()
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_file:
        fig_biplot.write_image(tmp_file.name)
        image_paths.append(tmp_file.name)

    # Loadings Heatmap
    fig_heatmap, ax_heatmap = plt.subplots(figsize=(12,8))
    sns.heatmap(
        loadings_df,
        cmap="coolwarm",
        center=0,
        annot=True,
        fmt=".2f",
        ax=ax_heatmap
    )
    ax_heatmap.set_title("Loadings Heatmap")
    plt.tight_layout()
    plt.show()
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_file:
        fig_heatmap.savefig(tmp_file.name, bbox_inches='tight')
        image_paths.append(tmp_file.name)
    plt.close(fig_heatmap)

    # Contribution Plots (Top 10 for each PC)
    for pc in loadings_df.columns:
        contrib = (
            loadings_df[pc]
            .abs()
            .sort_values(ascending=False)
            .head(10)
        )
        fig_contrib, ax_contrib = plt.subplots(figsize=(12,5))
        contrib.plot(kind='bar', ax=ax_contrib)
        ax_contrib.set_title(f"Contribuciones Top 10 - {pc}")
        ax_contrib.set_ylabel("Absolute Loading")
        ax_contrib.tick_params(axis='x', rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_file:
            fig_contrib.savefig(tmp_file.name, bbox_inches='tight')
            image_paths.append(tmp_file.name)
        plt.close(fig_contrib)

    return image_paths

def _perform_clustering(score_df, n_clusters_kmeans, min_cluster_size_hdbscan):
    """Realiza clustering K-Means y HDBSCAN."""
    print("\nRealizando Clustering...")
    image_paths = [] # List to store paths of saved images

    # Outliers Hotelling T²
    if score_df.shape[0] > 1 and score_df.shape[1] > 1: # Ensure enough data for covariance
        try:
            # Filter for numeric columns for covariance calculation
            numeric_score_cols = score_df.select_dtypes(include=np.number).columns.tolist()
            if 'T2' in numeric_score_cols: # Avoid using 'T2' itself if already present
                numeric_score_cols.remove('T2')

            if len(numeric_score_cols) > 1:
                cov = EmpiricalCovariance().fit(score_df[numeric_score_cols])
                mahal = cov.mahalanobis(score_df[numeric_score_cols])
                score_df["T2"] = mahal

                fig_t2 = px.scatter(
                    score_df,
                    x="PC1",
                    y="PC2",
                    color="T2",
                    title="Outlier Map - Hotelling T²"
                )
                fig_t2.show()
                with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_file:
                    fig_t2.write_image(tmp_file.name)
                    image_paths.append(tmp_file.name)
            else:
                print(f"Advertencia: Datos insuficientes para calcular Hotelling T² (se requieren al menos 2 columnas numéricas). Columnas disponibles: {numeric_score_cols}")
        except Exception as e:
            print(f"Advertencia: No se pudo calcular Hotelling T² ({e}). Posiblemente datos insuficientes o columnas constantes.")
    else:
        print("Advertencia: Datos insuficientes para calcular Hotelling T².")

    # K-Means
    try:
        # Filter for numeric PC columns for clustering
        pc_cols_for_kmeans = [col for col in score_df.columns if col.startswith('PC') and pd.api.types.is_numeric_dtype(score_df[col])]
        if len(pc_cols_for_kmeans) > 1: # K-Means typically needs at least 2 dimensions
            kmeans = KMeans(
                n_clusters=n_clusters_kmeans,
                random_state=42,
                n_init='auto'
            )
            score_df["KMEANS"] = (
                kmeans.fit_predict(score_df[pc_cols_for_kmeans])
                .astype(str)
            )
            fig_kmeans = px.scatter(
                score_df,
                x="PC1",
                y="PC2",
                color="KMEANS",
                title=f"K-Means Clusters (k={n_clusters_kmeans})"
            )
            fig_kmeans.show()
            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_file:
                fig_kmeans.write_image(tmp_file.name)
                image_paths.append(tmp_file.name)
        else:
            print(f"Advertencia: Datos insuficientes para K-Means (se requieren al menos 2 PCs numéricas). PCs disponibles: {pc_cols_for_kmeans}")
    except Exception as e:
        print(f"Advertencia: No se pudo realizar K-Means ({e}).")

    # HDBSCAN
    try:
        # Filter for numeric PC columns for clustering
        pc_cols_for_hdbscan = [col for col in score_df.columns if col.startswith('PC') and pd.api.types.is_numeric_dtype(score_df[col])]
        if len(pc_cols_for_hdbscan) > 1 and score_df.shape[0] >= min_cluster_size_hdbscan: # Ensure enough samples and dimensions for HDBSCAN
            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size_hdbscan
            )
            score_df["HDBSCAN"] = (
                clusterer.fit_predict(score_df[pc_cols_for_hdbscan])
                .astype(str)
            )
            fig_hdbscan = px.scatter(
                score_df,
                x="PC1",
                y="PC2",
                color="HDBSCAN",
                title=f"HDBSCAN Clusters (min_cluster_size={min_cluster_size_hdbscan})"
            )
            fig_hdbscan.show()
            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_file:
                fig_hdbscan.write_image(tmp_file.name)
                image_paths.append(tmp_file.name)
        elif len(pc_cols_for_hdbscan) < 2:
            print(f"Advertencia: Datos insuficientes para HDBSCAN (se requieren al menos 2 PCs numéricas). PCs disponibles: {pc_cols_for_hdbscan}")
        else:
             print(f"Advertencia: Datos insuficientes para HDBSCAN. Se requieren al menos {min_cluster_size_hdbscan} muestras, se tienen {score_df.shape[0]}.")
    except Exception as e:
        print(f"Advertencia: No se pudo realizar HDBSCAN ({e}).")

    return score_df, image_paths # Return image_paths from clustering plots


## 9. Función Principal de Ejecución de PCA

In [ ]:
def run_pca_analysis(b):
    clear_output(wait=True)

    print("\nINICIANDO ANÁLISIS PCA...\n")

    if df.empty:
        print("El DataFrame está vacío. Por favor, cargue un archivo CSV primero.")
        return

    mineral_cols = list(multi_select.value)
    if not mineral_cols:
        print("Por favor, seleccione al menos una variable para el PCA.")
        return

    epsilon = epsilon_widget.value
    n_components = ncomp_widget.value
    n_clusters_kmeans = kmeans_clusters_widget.value
    min_cluster_size_hdbscan = hdbscan_min_cluster_size_widget.value

    X = df[mineral_cols].copy()

    all_plot_image_paths = [] # Initialize list for all plot paths

    # 1. QA/QC y Preprocesamiento
    X_processed = _perform_qaqc(X, epsilon)

    # Ensure X_processed is not empty after QA/QC
    if X_processed.empty or X_processed.shape[1] < n_components:
        print("Advertencia: El DataFrame procesado está vacío o tiene menos columnas que el número de componentes. Ajustando n_components.")
        if X_processed.shape[1] == 0:
            print("No hay columnas válidas para PCA después del preprocesamiento.")
            return
        else:
            n_components = min(n_components, X_processed.shape[1])
            print(f"n_components ajustado a {n_components}")

    # 2. Cierre Composicional (CLR) y Escalado
    X_scaled, _ = _apply_clr_scaling(X_processed)

    # 3. Ejecución de PCA
    scores, loadings, explained, _ = _execute_pca(X_scaled, n_components, mineral_cols)

    # 4. Preparar DataFrame de Scores
    optional_widgets_values = {
        'lithology': lith_widget.value,
        'alteration': alter_widget.value,
        'depth': depth_widget.value,
        'domain': domain_widget.value
    }
    # Align score_df index to original_df_processed if needed
    score_df = _prepare_score_df(scores, n_components, X_processed.reset_index(drop=True), optional_widgets_values)

    # 5. Generar Plots de PCA
    pca_plot_paths = _generate_plots(score_df, explained, loadings, mineral_cols)
    all_plot_image_paths.extend(pca_plot_paths)

    # 6. Realizar Clustering y obtener paths de los plots de clustering
    score_df, clustering_plot_paths = _perform_clustering(score_df, n_clusters_kmeans, min_cluster_size_hdbscan)
    all_plot_image_paths.extend(clustering_plot_paths)

    # Guardar resultados en el contexto global para PDF
    global last_explained_df, last_loadings_df, last_score_df, last_image_paths
    last_explained_df = explained
    last_loadings_df = loadings
    last_score_df = score_df
    last_image_paths = all_plot_image_paths # Store all generated image paths

    print("\nCSV EXPORTADOS:")
    explained.to_csv(
        "explained_variance.csv",
        index=False
    )
    loadings.to_csv(
        "loadings.csv"
    )
    score_df.to_csv(
        "scores.csv",
        index=False
    )
    print("- explained_variance.csv")
    print("- loadings.csv")
    print("- scores.csv")

    print("\nANÁLISIS PCA FINALIZADO. Use el botón 'Generar PDF' para descargar el reporte.\n")


## 10. Botones de Ejecución

In [ ]:
# Variables globales para almacenar los últimos resultados del PCA para el PDF
last_explained_df = None
last_loadings_df = None
last_score_df = None
last_image_paths = [] # New global variable to store image paths

run_pca_button = widgets.Button(
    description="EJECUTAR PCA",
    button_style='success',
    disabled=df.empty
)

generate_pdf_button = widgets.Button(
    description="GENERAR PDF",
    button_style='info',
    disabled=True # Disable initially, enable after PCA run
)

output_area = widgets.Output()

def on_run_pca_clicked(b):
    with output_area:
        run_pca_analysis(b)
        # Enable PDF button after PCA is run and data is available
        if last_explained_df is not None:
            generate_pdf_button.disabled = False

def on_generate_pdf_clicked(b):
    with output_area:
        if last_explained_df is not None and last_loadings_df is not None and last_score_df is not None and last_image_paths:
            export_pdf_report(
                explained_df=last_explained_df,
                loadings_df=last_loadings_df,
                score_df=last_score_df,
                image_paths=last_image_paths # Pass image paths to PDF function
            )
        else:
            print("Por favor, ejecute el PCA primero para generar los datos y las imágenes para el PDF.")

run_pca_button.on_click(on_run_pca_clicked)
generate_pdf_button.on_click(on_generate_pdf_clicked)

display(run_pca_button, generate_pdf_button, output_area)
